# Lecture 15 · Assignment 2. Dying ReLU: diagnose it, fix it, measure it

At the end of Lecture 15 half of our seeds failed. Some got stuck at a loss of exactly **14**, others at exactly **6.8**, and the
`dead_neurons` counts did not always explain why. The lecture closed with "implement Leaky ReLU and improve this training".
This assignment does that, and then goes further. You will tell the two failure modes apart from the predictions alone,
separate *dead* neurons from neurons that merely receive *no gradient*, find out when a dead neuron can come back,
and judge every fix by how often it works rather than by one lucky seed.

**What you will do**
1. Add a `leaky_relu` operation to `Scalar` and gradient-check it
2. Thread `act='leaky'`, `alpha` and a linear output layer through `Neuron` and `MLP` without changing any seed's weights
3. Derive the two "stuck" losses and write a `diagnose` function
4. Count neurons that get zero gradient, and explain why that number differs from the dead count
5. Track when neurons die and revive during training, and prove which layers can never revive
6. Measure success rates over 20 seeds and tune the leak `alpha`

Cells marked **Exercise** contain `raise NotImplementedError` for you to replace. Each one is followed by a
**check** cell that prints ✅, ❌ or ⏳. Cells marked **Your answer** want a few sentences of explanation. Those are
graded too, and they matter as much as the code. Run the notebook from top to bottom. A later exercise often uses
the function you wrote in an earlier one. Parts 5 and 6 train many networks. Expect a few minutes of run time in total.

In [ ]:
import math, random

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

## Setup: the code from Lecture 15

`Scalar`, `Neuron`, `Layer`, `MLP`, the reorder data, `compute_loss`, `train` and `dead_neurons`, exactly as in the lecture.
We also include `gradcheck_fn`, a small numerical gradient checker for plain functions.

In [ ]:
class Scalar:
    def __init__(self, value, label='', oprt='', children=()):
        self.value = value
        self.label = label
        self.oprt = oprt
        self.children = children
        self.der = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value + other.value, '', oprt='+', children=(self, other))
        def _backward():
            self.der += 1.0 * out.der
            other.der += 1.0 * out.der
        out._backward = _backward
        return out

    def __radd__(self, other):          # other + self
        return self + other

    def __rmul__(self, other):          # other * self
        return self * other

    def __pow__(self, n):
        out = Scalar(self.value ** n, label='', children=(self,), oprt=f'**{n}')
        def _backward():
            self.der += n * (self.value ** (n-1)) * out.der
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):          # other - self
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value * other.value, '', oprt='*', children=(self, other))
        def _backward():
            self.der += other.value * out.der
            other.der += self.value * out.der
        out._backward = _backward
        return out

    def relu(self):
        out = Scalar(0 if self.value < 0 else self.value, '', oprt='ReLU', children=(self,))
        def _backward():
            self.der += (out.value > 0) * out.der
        out._backward = _backward
        return out

    def backward(self):
        visited = set()
        topo = list()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.der = 1.0
        for v in reversed(topo):
            v._backward()

In [ ]:
class Neuron:
    def __init__(self, n_inputs, act='relu', initial_bias=None):
        self.weights = [Scalar(random.uniform(-1,1), label=f'w{i+1}') for i in range(n_inputs)]
        self.bias = Scalar(random.uniform(-1,1), label='b') if initial_bias == None else Scalar(initial_bias)
        self.act = act
        self.last_pre = None              # pre-activation of the latest call, for debugging

    def __call__(self, x):
        n = sum(xi * wi for xi, wi in zip(x, self.weights)) + self.bias
        self.last_pre = n.value
        if self.act == 'relu':
            return n.relu()
        return n                          # linear

    @property
    def params(self):
        return self.weights + [self.bias]


class Layer:
    def __init__(self, n_inputs, n_outputs, **kwargs):
        self.neurons = [Neuron(n_inputs, **kwargs) for _ in range(n_outputs)]

    def __call__(self, x):
        outputs = [n(x) for n in self.neurons]
        return outputs[0] if len(outputs) == 1 else outputs

    @property
    def params(self):
        return [p for neuron in self.neurons for p in neuron.params]


class MLP:
    def __init__(self, n_inputs, output_layers, act='relu', initial_bias=None):
        sizes = [n_inputs] + output_layers
        self.layers = [Layer(sizes[i], sizes[i+1], act=act, initial_bias=initial_bias)
                       for i in range(len(output_layers))]

    def __call__(self, x):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def params(self):
        return [p for l in self.layers for p in l.params]

In [ ]:
# x = [demand, stock, incoming],   y (order) = max(0, demand - stock - incoming)
xs = [
    [3.0, 1.0, 1.0],   # 3 - 1 - 1 =  1
    [4.0, 1.0, 0.0],   # 4 - 1 - 0 =  3
    [2.0, 2.0, 1.0],   # 2 - 2 - 1 = -1  -> 0
    [5.0, 2.0, 1.0],   # 5 - 2 - 1 =  2
    [1.0, 0.0, 2.0],   # 1 - 0 - 2 = -1  -> 0
]
ys = [1.0, 3.0, 0.0, 2.0, 0.0]

def compute_loss(nn):
    ypred = [nn(x) for x in xs]
    return sum((ya - yp)**2 for ya, yp in zip(ys, ypred))

def train(model, lr=0.01, epochs=500, zero_grad=True):
    loss_history = []
    for step in range(epochs):
        loss = compute_loss(model)                       # 1. forward pass
        if zero_grad:
            for p in model.params: p.der = 0.0           # 2. zero grad
        loss.backward()                                  # 3. backward pass
        for p in model.params: p.value += -p.der * lr    # 4. update
        loss_history.append(loss.value)
    return loss_history

In [ ]:
def dead_neurons(model):
    """ per layer: how many neurons had pre-activation <= 0 for every training input """
    alive = {id(n): False for layer in model.layers for n in layer.neurons}
    for x in xs:
        model(x)                              # forward pass fills last_pre
        for layer in model.layers:
            for n in layer.neurons:
                if n.last_pre > 0:
                    alive[id(n)] = True       # fired at least once
    return [sum(not alive[id(n)] for n in layer.neurons) for layer in model.layers]

In [ ]:
def rel_error(a, b):
    return abs(a - b) / max(1.0, abs(a), abs(b))

def gradcheck_fn(f, values, h=1e-6, tol=1e-5):
    """ True if backprop agrees with central differences for f(list of Scalars) -> Scalar """
    vs = [Scalar(v) for v in values]
    f(vs).backward()
    for i, v in enumerate(vs):
        plus  = [Scalar(x + h if j == i else x) for j, x in enumerate(values)]
        minus = [Scalar(x - h if j == i else x) for j, x in enumerate(values)]
        numerical = (f(plus).value - f(minus).value) / (2 * h)
        if rel_error(v.der, numerical) >= tol:
            return False
    return True

## Part 1. The Leaky ReLU operation

$$\text{LeakyReLU}_\alpha(x) = \begin{cases} x & x > 0 \\ \alpha x & x \le 0 \end{cases} \qquad
\frac{d}{dx}\text{LeakyReLU}_\alpha(x) = \begin{cases} 1 & x > 0 \\ \alpha & x \le 0 \end{cases}$$

### Exercise 1.1

Write `leaky_relu(self, alpha=0.01)` following the same recipe as `relu`. It should compute the output, set `oprt='LeakyReLU'` and
`children=(self,)`, and define a `_backward` that **adds** the local derivative times `out.der` into `self.der`. The line after the
function attaches it to the class, so afterwards `s.leaky_relu()` works on any `Scalar`.

In [ ]:
def leaky_relu(self, alpha=0.01):
    # YOUR CODE HERE
    raise NotImplementedError

Scalar.leaky_relu = leaky_relu

In [ ]:
check("forward, positive and negative", lambda: Scalar(2.0).leaky_relu().value == 2.0 and
      abs(Scalar(-3.0).leaky_relu(0.1).value - (-0.3)) < 1e-12)
check("graph bookkeeping", lambda: (lambda s: (lambda o: o.oprt == 'LeakyReLU' and o.children == (s,))(s.leaky_relu()))(Scalar(1.0)))
check("gradient on both sides, two alphas", lambda: all(
      gradcheck_fn(lambda v: (v[0] * v[1] + 0.5).leaky_relu(a), [x, 1.3]) for a in (0.01, 0.3) for x in (0.7, -0.9)))
check("accumulates with +=", lambda: (lambda x: (x.leaky_relu(0.2) + x.leaky_relu(0.2)).backward() or abs(x.der - 0.4) < 1e-12)(Scalar(-1.0)))

**Your answer.** What derivative does your code use at exactly `x = 0`, and why does that choice not matter in practice? Then say in one sentence why a Leaky ReLU neuron can never be *permanently* dead.

_(write here)_

## Part 2. Plumbing: new options without changing any seed

### Exercise 2.1

Rewrite `Neuron` and `MLP` (you may copy the lecture versions and edit them) so that:

* `Neuron(n_inputs, act='relu', initial_bias=None, alpha=0.01)` accepts `act` equal to `'relu'`, `'leaky'` or `'linear'`, and stores `alpha`.
  `'leaky'` uses `n.leaky_relu(self.alpha)`.
* `MLP(n_inputs, output_layers, act='relu', linear_output=False, **kwargs)` gives the **last** layer `act='linear'` when
  `linear_output=True`, and passes every other keyword (`initial_bias`, `alpha`) down to each `Neuron`.
* `initial_bias` becomes a proper trainable `Scalar` with `label='b'`.

**Hard requirement.** `random.seed(s); MLP(3, [4, 4, 1], ...)` must produce *exactly* the same initial weights as the lecture's
`MLP` for every seed and every option, because we will compare our results with the lecture's tables seed by seed.
So you must call `random.uniform` the same number of times and in the same order as before.

In [ ]:
class Neuron:
    # YOUR CODE HERE
    def __init__(self, *args, **kwargs):
        raise NotImplementedError

class MLP:
    # YOUR CODE HERE
    def __init__(self, *args, **kwargs):
        raise NotImplementedError

In [ ]:
def lecture_initial_params(seed, sizes=(3, 4, 4, 1)):
    """ the initial parameter values the lecture's MLP draws for this seed: per neuron, weights then bias """
    random.seed(seed)
    values = []
    for n_in, n_out in zip(sizes, sizes[1:]):
        for _ in range(n_out):
            values += [random.uniform(-1, 1) for _ in range(n_in)] + [random.uniform(-1, 1)]
    return values

def test_same_seeds():
    ok = True
    for seed in range(5):
        for kwargs in ({}, {'act': 'leaky'}, {'act': 'leaky', 'alpha': 0.2, 'linear_output': True}):
            random.seed(seed)
            ok &= [p.value for p in MLP(3, [4, 4, 1], **kwargs).params] == lecture_initial_params(seed)
    return ok

def test_options():
    random.seed(0)
    m = MLP(3, [4, 4, 1], act='leaky', alpha=0.2, linear_output=True)
    acts = [n.act for l in m.layers for n in l.neurons]
    ok = acts == ['leaky'] * 8 + ['linear'] and all(n.alpha == 0.2 for l in m.layers for n in l.neurons)
    n = m.layers[0].neurons[0]
    for p, v in zip(n.params, [1.0, 1.0, 1.0, -10.0]):
        p.value = v
    out = n([1.0, 2.0, 3.0])                       # pre-activation 1+2+3-10 = -4  -> leaky gives -0.8
    ok &= abs(out.value - (-0.8)) < 1e-12
    random.seed(0)
    m2 = MLP(3, [4, 4, 1], initial_bias=0.1)
    b = m2.layers[0].neurons[0].bias
    ok &= isinstance(b, Scalar) and b.value == 0.1 and b.label == 'b' and len(m2.params) == 41
    return ok

check("same initial weights as the lecture for every seed and option", test_same_seeds)
check("act, alpha, linear_output and initial_bias work", test_options)

## Part 3. Reading the loss

In the lecture two numbers kept appearing. Seeds that died completely ended at a loss of exactly **14**. Others ended at
exactly **6.8**.

### Exercise 3.1. Derive the stuck losses

Write `stuck_losses(targets)` returning a pair:

1. the loss when the network predicts **0** for every row, and
2. the loss when the network predicts the **same constant** `c` for every row, using the `c` that gradient descent would settle on.

Use formulas in terms of `targets`. Do not hard-code 14 or 6.8.

In [ ]:
def stuck_losses(targets):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("stuck losses for the lecture data", lambda: all(abs(a - b) < 1e-12 for a, b in zip(stuck_losses(ys), (14.0, 6.8))))
check("stuck losses for other targets", lambda: all(abs(a - b) < 1e-12 for a, b in zip(stuck_losses([2.0, 4.0, 0.0]), (20.0, 8.0))))

**Your answer.** Prove that the best constant prediction is the mean of the targets. Then explain *which* part of the network must have died for the network to predict a constant, and why the constant it learns is the mean and not some other number.

_(write here)_

### Exercise 3.2. `diagnose`

Write `diagnose(model)` that runs the model on the five training rows and returns one of these strings:

* `'converged'` if the loss is below `0.05`
* `'all zero'` if every prediction is within `1e-3` of 0
* `'constant'` if all predictions are within `1e-3` of each other (but not all zero)
* `'other'` otherwise

Decide from the **predictions**, not from the loss value. The check then confirms that your labels line up with the two stuck
losses from Exercise 3.1.

In [ ]:
def diagnose(model):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_diagnose():
    zero_loss, const_loss = stuck_losses(ys)
    ok = True
    for seed in range(20):
        random.seed(seed)
        model = MLP(3, [4, 4, 1])
        train(model)
        label = diagnose(model)
        loss = compute_loss(model).value
        print(f"   seed {seed:2d}  loss {loss:8.4f}  {label:10s}  dead_neurons {dead_neurons(model)}")
        ok &= (label == 'all zero') == (abs(loss - zero_loss) < 1e-3)
        ok &= (label == 'constant') == (abs(loss - const_loss) < 1e-3)
    return ok

check("diagnose agrees with the stuck losses on 20 seeds", test_diagnose)

### Exercise 3.3. A puzzle

Look at seeds 5, 11, 15 and 19 in the table above. Their loss is 14, so the network predicts (almost) 0 everywhere, yet
`dead_neurons` says the **output neuron is not dead** (last entry 0). Run the cell below, which prints the output neuron's
pre-activation on every training row together with the target, then answer.

In [ ]:
for seed in (5, 15):
    random.seed(seed)
    model = MLP(3, [4, 4, 1])
    train(model)
    print(f"seed {seed}")
    for x, y in zip(xs, ys):
        model(x)
        print(f"   x {x}  target {y}  output pre-activation {model.layers[-1].neurons[0].last_pre:+.6f}")

**Your answer.** Why is the output neuron "alive" by the `dead_neurons` test, yet the network behaves as if it were dead? Why will training never get it out of this state? Suggest a better definition of a dead neuron.

_(write here)_

## Part 4. Dead versus starved

A neuron can stop learning for two different reasons. It can be **dead**, with pre-activation <= 0 on every row so that its own ReLU
blocks the gradient. Or it can be **starved**: it is fine itself, but nothing downstream sends any gradient back to it.

### Exercise 4.1

Write `zero_grad_neurons(model)`. It computes the full training loss, zeroes all `der`s, calls `backward()`, and returns for each layer
the number of neurons whose parameters (all weights **and** the bias) have `der` exactly `0.0`.

In [ ]:
def zero_grad_neurons(model):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_zero_grad_neurons():
    random.seed(0)
    m = MLP(3, [4, 4, 1])
    train(m)                                    # seed 0 ends with a dead output neuron
    a = zero_grad_neurons(m) == [4, 4, 1]
    random.seed(9)
    m = MLP(3, [4, 4, 1])
    train(m)
    b = zero_grad_neurons(m) == [2, 2, 0]
    return a and b

check("zero_grad_neurons", test_zero_grad_neurons)

print(f"{'':12}{'ReLU: dead':>14}{'zero-grad':>14}   |{'Leaky: dead':>14}{'zero-grad':>14}")
for seed in range(10):
    row = []
    for act, steps in (('relu', 500), ('leaky', 1000)):
        random.seed(seed)
        m = MLP(3, [4, 4, 1], act=act)
        train(m, epochs=steps)
        row += [dead_neurons(m), zero_grad_neurons(m)]
    print(f"seed {seed:2d}    {str(row[0]):>14}{str(row[1]):>14}   |{str(row[2]):>14}{str(row[3]):>14}")

**Your answer.** (a) For ReLU, prove that in every layer zero-grad >= dead. Pick one seed from the table where zero-grad is strictly larger in layer 0, and explain which neurons are starved and why. (b) For Leaky ReLU the "dead" counts are as large as for ReLU, but the zero-grad counts are all 0. Explain why both observations are true at once, and which of the two numbers predicts whether training succeeds.

_(write here)_

## Part 5. Can the dead come back?

### Exercise 5.1. `revival_events`

Write `revival_events(seed, lr=0.01, steps=500, **mlp_kwargs)`. It builds `random.seed(seed); MLP(3, [4, 4, 1], **mlp_kwargs)` and trains it
with the usual loop (forward, zero grad, backward, update) for `steps` updates. **Before the first update and after every
update** it records which neurons are alive, meaning `last_pre > 0` on at least one training row. It returns a list of
`(layer, neuron, step)` for every time a neuron that was dead after `step - 1` updates is alive after `step` updates.

In [ ]:
def revival_events(seed, lr=0.01, steps=500, **mlp_kwargs):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_revivals():
    e7, e0, e5 = revival_events(7), revival_events(0), revival_events(5)
    print("   seed 7:", e7, "  seed 0:", e0, "  seed 5:", e5)
    return e7 == [(1, 1, 5)] and e0 == [] and e5 == [(1, 0, 1), (1, 1, 1), (1, 2, 1)]

check("revival_events", test_revivals)

layer_counts = [0, 0, 0]
for seed in range(10):
    for layer, neuron, step in revival_events(seed):
        layer_counts[layer] += 1
print("revivals per layer over seeds 0-9 (ReLU, lr=0.01):", layer_counts)
print("seed 1, first 8 events:", revival_events(1)[:8])

**Your answer.** (a) Prove that with plain ReLU a neuron in the **first** hidden layer that is dead can never come back, no matter how long we train. (b) Why does that proof break for the second hidden layer? (c) Seed 1 shows the same layer-1 neuron reviving again and again a few steps apart. What is happening to it?

_(write here)_

## Part 6. Judge fixes by success rate

One seed proves nothing. A method should be judged by **how often** it works from a random start.

### Exercise 6.1. `success_rate`

Write `success_rate(title, seeds=range(20), lr=0.01, steps=500, **mlp_kwargs)`. For every seed it trains
`MLP(3, [4, 4, 1], **mlp_kwargs)` with `train(model, lr=lr, epochs=steps)` and then calls `diagnose(model)`. It prints one line and
returns a dict with the count of each `diagnose` label (`'converged'`, `'all zero'`, `'constant'`, `'other'`, all four keys present)
plus `'dead_hidden'`, the average total number of dead neurons in the two hidden layers after training.

In [ ]:
def success_rate(title, seeds=range(20), lr=0.01, steps=500, **mlp_kwargs):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_success_rate():
    r = success_rate("check: baseline seeds 0-9", seeds=range(10))
    return (set(r) == {'converged', 'all zero', 'constant', 'other', 'dead_hidden'}
            and r['converged'] == 7 and r['all zero'] == 2 and r['constant'] == 1)

check("success_rate", test_success_rate)

Now the full comparison. It takes about two minutes.

In [ ]:
results = {}
results['baseline'] = success_rate("Baseline ReLU, lr=0.01")
results['big lr']   = success_rate("Big lr=0.1", lr=0.1)
results['small lr'] = success_rate("Fix 1: lr=0.003, 1500 steps", lr=0.003, steps=1500)
results['bias 0.1'] = success_rate("Try: initial_bias=0.1", initial_bias=0.1)
results['leaky']    = success_rate("Fix 2: Leaky ReLU, 1000 steps", act='leaky', steps=1000)
results['linear']   = success_rate("Fix 3: linear output, 1000 steps", linear_output=True, steps=1000)
results['all']      = success_rate("All fixes", act='leaky', linear_output=True, lr=0.003, steps=1500)

**Your answer.** Using the failure-mode columns, explain for **each** fix which failure mode it removes and which it does not. In particular: why does the linear output remove every `all zero` failure but not every `constant` one? Why is `initial_bias=0.1` worse than doing nothing, even though it sounds like it should keep neurons alive?

_(write here)_

### Exercise 6.2. How big should the leak be?

The default `alpha = 0.01` is a convention. Write `alpha_sweep(alphas, seeds=range(20), steps=1000)`. For each `alpha` it trains
`MLP(3, [4, 4, 1], act='leaky', alpha=alpha)` with `lr=0.01` on every seed and returns
`{alpha: (n_converged, median_steps)}`. Here `median_steps` is the median, over the converged seeds only, of the first step
at which the training loss dropped below `0.05`. Use `statistics.median`, and return `None` for the median if nothing converged.

In [ ]:
import statistics

def alpha_sweep(alphas, seeds=range(20), steps=1000):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
sweep = alpha_sweep([0.01, 0.1, 0.3])
for alpha, (n_ok, med) in sweep.items():
    print(f"alpha {alpha:<5} converged {n_ok:2d}/20   median steps to loss < 0.05: {med}")
check("alpha_sweep returns the right shape", lambda: set(sweep) == {0.01, 0.1, 0.3}
      and all(isinstance(n, int) and 0 <= n <= 20 for n, _ in sweep.values()))

**Your answer.** Describe the trade-off you see between a small and a large `alpha`. What happens in the limit `alpha = 1`, and why would that network be unable to learn `max(0, demand - stock - incoming)` at all?

_(write here)_